# Component 7: Receding Horizon Control

Diffusion Policy doesn't just predict actions -- it predicts an entire **trajectory** of 16 future actions. But it doesn't blindly execute all 16. Instead, it uses **receding horizon control**:

1. **Plan**: Predict 16 actions into the future
2. **Execute**: Only execute the first 8 actions
3. **Re-plan**: From the new position, predict 16 more actions
4. **Repeat**

## Why not execute all 16?

The world changes! By the time you've executed 8 actions, new information is available:
- The object may have moved
- The robot's actual position may differ from predicted (due to dynamics)
- A human may have intervened

Re-planning lets the policy **adapt** while maintaining **temporal consistency** (the first few actions of each plan overlap with the previous plan).

## The Balance

| Execute fewer steps | Execute more steps |
|---|---|
| More adaptive | More committed |
| More computation | Less computation |
| Potentially jittery | Smoother but less reactive |
| Good for dynamic environments | Good for stable environments |

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Diffusion Policy configuration values (from the default PushT config)
# These are the standard values used in the diffusion policy paper.
# We define them here directly instead of loading the full lerobot package.
# ============================================================

class DiffusionPolicyConfig:
    """Standard diffusion policy configuration for PushT."""
    horizon = 16          # Total prediction horizon (number of future actions predicted)
    n_action_steps = 8    # Number of actions actually executed before re-planning
    n_obs_steps = 2       # Number of observation frames used as context
    action_dim = 2        # PushT action dimension (x, y)

cfg = DiffusionPolicyConfig()

print("Diffusion Policy Configuration (PushT defaults)")
print("=" * 50)
print(f"  Prediction horizon:     {cfg.horizon} steps")
print(f"  Action steps executed:  {cfg.n_action_steps} steps")
print(f"  Observation steps:      {cfg.n_obs_steps} frames")
print(f"  Action dimension:       {cfg.action_dim}d (x, y)")
print(f"\nThis means the policy predicts {cfg.horizon} actions but only")
print(f"executes {cfg.n_action_steps} before re-planning.")

## Simulating Receding Horizon Control

We don't need a real robot or environment to understand receding horizon. Let's simulate it with synthetic trajectories. The key idea:

- A **planner** generates a smooth 16-step trajectory from the current position to a target
- We **execute** only the first `n_execute` steps
- We **re-plan** from the new position
- Each plan overlaps with the previous one

In [ ]:
# ============================================================
# Synthetic trajectory planner
# ============================================================

def simulate_plan(start_pos, target_pos, n_steps=16):
    """Generate a smooth trajectory from start to target.
    
    Uses cubic interpolation with a slight sinusoidal curvature
    to simulate realistic robot motion planning.
    """
    t = np.linspace(0, 1, n_steps)
    # Cubic ease-in-out for more realistic acceleration/deceleration
    t_smooth = 3 * t**2 - 2 * t**3
    
    trajectory = np.zeros((n_steps, 2))
    trajectory[:, 0] = start_pos[0] + (target_pos[0] - start_pos[0]) * t_smooth
    trajectory[:, 1] = start_pos[1] + (target_pos[1] - start_pos[1]) * t_smooth + 0.1 * np.sin(np.pi * t)
    return trajectory


# ============================================================
# Run 5 planning cycles with receding horizon
# ============================================================

n_plan = 16      # Plan this many steps ahead
n_execute = 8    # Execute only the first 8
n_cycles = 5     # Number of planning cycles

target = np.array([5.0, 3.0])    # Fixed target position
current_pos = np.array([0.0, 0.0])  # Starting position

# Store all plans and executed segments
all_plans = []
all_executed = []
all_positions = [current_pos.copy()]

print("=" * 60)
print(f"Receding Horizon: Plan {n_plan}, Execute {n_execute}")
print("=" * 60)
print(f"Target: {target}")
print(f"Start:  {current_pos}")
print()

for cycle in range(n_cycles):
    # PLAN: Generate a full 16-step trajectory
    plan = simulate_plan(current_pos, target, n_steps=n_plan)
    all_plans.append(plan.copy())
    
    # EXECUTE: Only take the first n_execute steps
    executed = plan[:n_execute]
    all_executed.append(executed.copy())
    
    # Move to the end of the executed segment
    current_pos = executed[-1].copy()
    all_positions.append(current_pos.copy())
    
    dist_to_target = np.linalg.norm(current_pos - target)
    print(f"Cycle {cycle + 1}: pos=({current_pos[0]:.2f}, {current_pos[1]:.2f}), "
          f"dist_to_target={dist_to_target:.3f}")

print(f"\nFinal position: ({current_pos[0]:.2f}, {current_pos[1]:.2f})")
print(f"Target:         ({target[0]:.2f}, {target[1]:.2f})")
print(f"Final error:    {np.linalg.norm(current_pos - target):.4f}")

In [ ]:
# ============================================================
# Visualize the full trajectory with all plan segments
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#1A1915')
fig.suptitle('Receding Horizon Control: Plan 16, Execute 8, Re-plan',
             fontsize=14, fontweight='bold', color='white')

# Color palette for different cycles
cycle_colors = ['#D97757', '#5B8CB8', '#7DA488', '#9B7EC8', '#C4956A']

# Left plot: 2D trajectory view
ax = axes[0]
ax.set_facecolor('#2A2A25')
ax.set_title('2D Trajectory (Bird\'s Eye View)', color='white', fontsize=12)

for i, (plan, executed) in enumerate(zip(all_plans, all_executed)):
    color = cycle_colors[i % len(cycle_colors)]
    
    # Draw the full plan (dashed, faded)
    ax.plot(plan[:, 0], plan[:, 1], '--', color=color, alpha=0.3, linewidth=1.5)
    ax.plot(plan[n_execute:, 0], plan[n_execute:, 1], 'x', color=color, alpha=0.3,
            markersize=4, label=f'Cycle {i+1} discarded' if i == 0 else None)
    
    # Draw the executed portion (solid, bright)
    ax.plot(executed[:, 0], executed[:, 1], 'o-', color=color, linewidth=2.5,
            markersize=5, alpha=0.9, label=f'Cycle {i+1} executed')
    
    # Highlight the overlap region (last executed step = first of next plan)
    if i < len(all_plans) - 1:
        ax.plot(executed[-1, 0], executed[-1, 1], 'D', color='white',
                markersize=8, zorder=5, alpha=0.8)

# Start and target
ax.plot(0, 0, '*', color='#7DA488', markersize=15, zorder=6, label='Start')
ax.plot(target[0], target[1], '*', color='#D97757', markersize=15, zorder=6, label='Target')

ax.set_xlabel('X Position', color='white')
ax.set_ylabel('Y Position', color='white')
ax.legend(fontsize=8, loc='upper left')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15)
ax.set_aspect('equal')
for spine in ax.spines.values():
    spine.set_color('gray')

# Right plot: Timeline view showing plan/execute windows
ax = axes[1]
ax.set_facecolor('#2A2A25')
ax.set_title('Timeline: Plan vs Execute Windows', color='white', fontsize=12)

for i, (plan, executed) in enumerate(zip(all_plans, all_executed)):
    color = cycle_colors[i % len(cycle_colors)]
    global_offset = i * n_execute  # Global timestep offset
    
    # Full plan window
    plan_start = global_offset
    plan_end = global_offset + n_plan
    ax.barh(i, n_plan, left=plan_start, height=0.4, color=color, alpha=0.25,
            edgecolor=color, linewidth=1.5, linestyle='--')
    
    # Executed window
    ax.barh(i, n_execute, left=plan_start, height=0.4, color=color, alpha=0.8,
            edgecolor='white', linewidth=1.5)
    
    # Labels
    ax.text(plan_start + n_execute/2, i, f'Execute {n_execute}', ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')
    ax.text(plan_start + n_execute + (n_plan - n_execute)/2, i, f'Discard {n_plan - n_execute}',
            ha='center', va='center', fontsize=7, color=color, alpha=0.7)

ax.set_xlabel('Global Timestep', color='white')
ax.set_ylabel('Planning Cycle', color='white')
ax.set_yticks(range(n_cycles))
ax.set_yticklabels([f'Cycle {i+1}' for i in range(n_cycles)])
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, axis='x')
ax.invert_yaxis()
for spine in ax.spines.values():
    spine.set_color('gray')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Compare: Execute-all-16 vs Execute-8-replan
# When the target MOVES mid-trajectory!
# ============================================================

def run_receding_horizon(start, targets, n_plan=16, n_execute=8, target_change_at=None):
    """Run receding horizon control with optional target change.
    
    Args:
        start: starting position [x, y]
        targets: list of (step, target_pos) tuples - target changes at given steps
        n_plan: planning horizon
        n_execute: execution horizon
        target_change_at: deprecated, use targets instead
    
    Returns:
        executed_trajectory: full trajectory of executed positions
        plans: list of all generated plans
    """
    current_pos = np.array(start, dtype=float)
    executed_trajectory = [current_pos.copy()]
    plans = []
    global_step = 0
    current_target_idx = 0
    
    n_total_cycles = 8
    
    for cycle in range(n_total_cycles):
        # Check if target should change
        while (current_target_idx < len(targets) - 1 and 
               global_step >= targets[current_target_idx + 1][0]):
            current_target_idx += 1
        
        current_target = np.array(targets[current_target_idx][1])
        
        # Plan
        plan = simulate_plan(current_pos, current_target, n_steps=n_plan)
        plans.append({'plan': plan, 'target': current_target.copy(), 'start_step': global_step})
        
        # Execute
        steps_to_execute = min(n_execute, n_plan)
        for step in range(steps_to_execute):
            executed_trajectory.append(plan[step].copy())
            global_step += 1
        
        current_pos = plan[steps_to_execute - 1].copy()
        
        # Check if close enough to target
        if np.linalg.norm(current_pos - current_target) < 0.05:
            break
    
    return np.array(executed_trajectory), plans


# Scenario: target moves at step 20
initial_target = [5.0, 3.0]
new_target = [2.0, 5.0]
target_change_step = 20

targets = [(0, initial_target), (target_change_step, new_target)]

# Strategy 1: Execute all 16 (re-plan every 16 steps)
traj_16, plans_16 = run_receding_horizon([0, 0], targets, n_plan=16, n_execute=16)

# Strategy 2: Execute 8, re-plan every 8 steps  
traj_8, plans_8 = run_receding_horizon([0, 0], targets, n_plan=16, n_execute=8)

# Strategy 3: Execute 4, re-plan every 4 steps (most adaptive)
traj_4, plans_4 = run_receding_horizon([0, 0], targets, n_plan=16, n_execute=4)

print("=" * 60)
print("Comparison: Different Execution Horizons")
print("=" * 60)
print(f"Target moves from {initial_target} to {new_target} at step {target_change_step}")
print(f"\nExecute-16: {len(traj_16)} total steps, {len(plans_16)} planning cycles")
print(f"Execute-8:  {len(traj_8)} total steps, {len(plans_8)} planning cycles")
print(f"Execute-4:  {len(traj_4)} total steps, {len(plans_4)} planning cycles")

# Final distances to NEW target
print(f"\nFinal distance to new target ({new_target}):")
print(f"  Execute-16: {np.linalg.norm(traj_16[-1] - new_target):.3f}")
print(f"  Execute-8:  {np.linalg.norm(traj_8[-1] - new_target):.3f}")
print(f"  Execute-4:  {np.linalg.norm(traj_4[-1] - new_target):.3f}")

In [ ]:
# ============================================================
# Visualize the comparison
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#1A1915')
fig.suptitle('Target Moves Mid-Trajectory: How Fast Does Each Strategy Adapt?',
             fontsize=14, fontweight='bold', color='white')

strategies = [
    ('Execute ALL 16', traj_16, plans_16, '#D97757'),
    ('Execute 8, Re-plan', traj_8, plans_8, '#5B8CB8'),
    ('Execute 4, Re-plan', traj_4, plans_4, '#7DA488'),
]

for idx, (title, traj, plans, color) in enumerate(strategies):
    ax = axes[idx]
    ax.set_facecolor('#2A2A25')
    ax.set_title(title, color='white', fontsize=12)
    
    # Draw executed trajectory
    ax.plot(traj[:, 0], traj[:, 1], 'o-', color=color, linewidth=2,
            markersize=3, alpha=0.9, label='Executed path')
    
    # Draw planned but discarded portions
    for p_info in plans:
        plan = p_info['plan']
        ax.plot(plan[:, 0], plan[:, 1], '--', color=color, alpha=0.15, linewidth=1)
    
    # Mark target change
    if target_change_step < len(traj):
        ax.plot(traj[target_change_step, 0], traj[target_change_step, 1],
                'X', color='yellow', markersize=12, zorder=5, label=f'Target moves (step {target_change_step})')
    
    # Start, initial target, new target
    ax.plot(0, 0, '*', color='white', markersize=12, zorder=6, label='Start')
    ax.plot(initial_target[0], initial_target[1], 'p', color='#C4956A',
            markersize=12, zorder=6, label='Initial target')
    ax.plot(new_target[0], new_target[1], 'p', color='#9B7EC8',
            markersize=12, zorder=6, label='New target')
    
    # Draw arrow from initial to new target
    ax.annotate('', xy=new_target, xytext=initial_target,
                arrowprops=dict(arrowstyle='->', color='yellow', lw=1.5, linestyle='--'))
    
    ax.set_xlabel('X', color='white')
    ax.set_ylabel('Y', color='white')
    ax.legend(fontsize=7, loc='lower right')
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.15)
    ax.set_xlim(-0.5, 6)
    ax.set_ylim(-0.5, 6)
    for spine in ax.spines.values():
        spine.set_color('gray')

plt.tight_layout()
plt.show()

print("\nKey observation: Shorter execution horizons adapt faster when the target moves,")
print("but they require more frequent replanning (more computation).")

In [ ]:
# ============================================================
# Disturbance response: target jumps suddenly
# ============================================================

def run_with_disturbance(start, initial_target, disturbance_step, new_target, n_execute=8):
    """Run receding horizon and measure response time to disturbance."""
    targets = [(0, initial_target), (disturbance_step, new_target)]
    traj, plans = run_receding_horizon(start, targets, n_plan=16, n_execute=n_execute)
    
    # Measure: how many steps after disturbance before heading toward new target?
    distances_to_new = [np.linalg.norm(pos - new_target) for pos in traj]
    
    # Find the step where distance starts decreasing (response begins)
    response_step = None
    for i in range(min(disturbance_step, len(distances_to_new) - 1), len(distances_to_new) - 1):
        if distances_to_new[i+1] < distances_to_new[i]:
            response_step = i - disturbance_step
            break
    
    return traj, plans, distances_to_new, response_step


# Run disturbance test for different execution horizons
execute_horizons = [4, 8, 12, 16]
disturbance_step = 16
results = {}

for n_exec in execute_horizons:
    traj, plans, dists, response = run_with_disturbance(
        [0, 0], [5, 3], disturbance_step, [1, 5], n_execute=n_exec)
    results[n_exec] = {'traj': traj, 'plans': plans, 'dists': dists, 'response': response}

print("=" * 60)
print("Disturbance Response Analysis")
print("=" * 60)
print(f"Disturbance at step {disturbance_step}: target jumps from [5,3] to [1,5]")
print()

for n_exec in execute_horizons:
    r = results[n_exec]
    resp = r['response']
    print(f"Execute-{n_exec}: response delay = {resp if resp is not None else 'N/A'} steps, "
          f"final dist = {r['dists'][-1]:.3f}")

# Distance over time plot
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
fig.patch.set_facecolor('#1A1915')
ax.set_facecolor('#2A2A25')
ax.set_title('Distance to NEW Target Over Time (Disturbance Response)',
             color='white', fontsize=13, fontweight='bold')

colors = ['#D97757', '#5B8CB8', '#7DA488', '#9B7EC8']
for i, n_exec in enumerate(execute_horizons):
    dists = results[n_exec]['dists']
    steps = np.arange(len(dists))
    ax.plot(steps, dists, 'o-', color=colors[i], linewidth=2, markersize=3,
            label=f'Execute-{n_exec}', alpha=0.9)

# Mark disturbance
ax.axvline(x=disturbance_step, color='yellow', linestyle='--', linewidth=2,
           alpha=0.7, label=f'Target moves (step {disturbance_step})')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 10],
                 disturbance_step, disturbance_step + 16,
                 alpha=0.1, color='yellow', label='Reaction window')

ax.set_xlabel('Global Timestep', color='white', fontsize=11)
ax.set_ylabel('Distance to New Target', color='white', fontsize=11)
ax.legend(fontsize=9)
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15)
for spine in ax.spines.values():
    spine.set_color('gray')

plt.tight_layout()
plt.show()

print("\nShorter execution horizons respond faster because they re-plan sooner.")
print("But they also require more frequent planning (more neural network forward passes).")

In [ ]:
# ============================================================
# Timing analysis: real-world implications
# ============================================================

print("=" * 60)
print("Timing Analysis: Real-World Implications")
print("=" * 60)

control_freq = 10  # Hz (10 FPS, typical for robot control)
dt = 1.0 / control_freq  # 0.1 seconds per step

print(f"\nControl frequency: {control_freq} Hz ({dt*1000:.0f} ms per step)")
print(f"Planning horizon:  {n_plan} steps = {n_plan * dt:.1f} seconds")
print()

configs = [
    (4, 'Very adaptive'),
    (8, 'Balanced (default)'),
    (12, 'Mostly committed'),
    (16, 'Fully committed'),
]

print(f"{'Execute':>10} | {'Commitment':>15} | {'Re-plan freq':>15} | {'Style':>20}")
print("-" * 70)
for n_exec, style in configs:
    commitment_time = n_exec * dt
    replan_freq = control_freq / n_exec
    print(f"{n_exec:>10} | {commitment_time:>13.1f} s | {replan_freq:>12.1f} Hz | {style:>20}")

# Visualize timing
fig, ax = plt.subplots(1, 1, figsize=(14, 4))
fig.patch.set_facecolor('#1A1915')
ax.set_facecolor('#2A2A25')
ax.set_title('Timing: Action Commitment per Planning Cycle', color='white',
             fontsize=13, fontweight='bold')

colors = ['#D97757', '#5B8CB8', '#7DA488', '#9B7EC8']
bar_height = 0.6

for i, (n_exec, style) in enumerate(configs):
    color = colors[i]
    
    # Draw 3 planning cycles
    for cycle in range(3):
        start_time = cycle * n_exec * dt
        
        # Full plan (faded)
        ax.barh(i, n_plan * dt, left=start_time, height=bar_height,
                color=color, alpha=0.15, edgecolor=color, linewidth=1)
        
        # Executed portion (solid)
        ax.barh(i, n_exec * dt, left=start_time, height=bar_height,
                color=color, alpha=0.8, edgecolor='white', linewidth=1)
        
        if cycle == 0:
            ax.text(start_time + n_exec * dt / 2, i,
                    f'{n_exec * dt:.1f}s', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')

ax.set_xlabel('Time (seconds)', color='white', fontsize=11)
ax.set_yticks(range(len(configs)))
ax.set_yticklabels([f'Exec-{n}\n({s})' for n, s in configs], fontsize=9, color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.15, axis='x')
ax.invert_yaxis()
for spine in ax.spines.values():
    spine.set_color('gray')

# Add time markers
for t in np.arange(0, 5, 0.5):
    ax.axvline(x=t, color='white', alpha=0.05, linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nSolid bars = committed execution. Faded bars = planned but may be discarded.")
print("Each re-plan requires a full diffusion denoising pass (~100 steps of DDPM or ~10 of DDIM).")

In [ ]:
# ============================================================
# Show this with the REAL policy's action chunking config
# ============================================================

print("=" * 60)
print("Real Diffusion Policy Configuration")
print("=" * 60)

print(f"\nHorizon (prediction length):   {cfg.horizon}")
print(f"Number of action steps:         {cfg.n_action_steps}")
print(f"Number of observation steps:    {cfg.n_obs_steps}")

print(f"\nThis means:")
print(f"  - The model predicts {cfg.horizon} actions per planning cycle")
print(f"  - Only {cfg.n_action_steps} actions are executed before re-planning")
print(f"  - {cfg.horizon - cfg.n_action_steps} actions are discarded each cycle")
print(f"  - The model uses {cfg.n_obs_steps} observation frames as context")

print(f"\nAt {control_freq} Hz:")
print(f"  - Planning looks {cfg.horizon * dt:.1f}s into the future")
print(f"  - Commits to {cfg.n_action_steps * dt:.1f}s of action")
print(f"  - Re-plans every {cfg.n_action_steps * dt:.1f}s")

print("\n" + "-" * 60)
print("Why this matters for real robots:")
print("-" * 60)
print("1. SMOOTHNESS: Predicting 16 steps ensures the planned trajectory")
print("   is smooth and physically plausible (no sudden jerks).")
print("2. ADAPTABILITY: Executing only 8 means we can correct for")
print("   prediction errors and environmental changes every 0.8s.")
print("3. TEMPORAL CONSISTENCY: Consecutive plans overlap, so the")
print("   transition between plans is smooth (no discontinuities).")

## Key Takeaways

1. **Plan long, execute short**: Predict 16 steps for smooth trajectories, but only commit to 8 for adaptability
2. **Re-planning is cheap insurance**: Each re-plan costs one diffusion pass, but prevents catastrophic errors from stale predictions
3. **The execute horizon is a tunable knob**: 4 steps = very reactive, 16 steps = very committed. The right value depends on how dynamic your environment is
4. **At 10 Hz with execute-8**: You commit to 0.8 seconds of action before reconsidering. This is fast enough for most manipulation tasks

## TODO Exercise

Try execute-4 vs execute-8 vs execute-12 with the disturbance simulation above.

- Which gives the **smoothest** trajectory when the target is stationary?
- Which gives the **fastest** adaptation when the target moves?
- Can you find a scenario where execute-4 actually performs **worse** than execute-8? (Hint: think about planning noise)
- Bonus: Add random noise to each plan (`+ np.random.randn(...) * 0.1`) and see how different execution horizons handle planning uncertainty